# S10 — Paired Phase 1–Phase 2 multiseed audit

Reproduces the executed paired-seed robustness comparison using matched checkpoint-selection rules. It is supplementary evidence, not the source of the reported production metrics.

The public copy is output-stripped; authoritative exported tables and figures are distributed separately in the repository.

In [ ]:
from pathlib import Path
import gc, hashlib, importlib.util, json, sys
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
PUBLICATION = PROJECT.parent
ROOT = PROJECT / "Ablations" / "PreSubmission_Independent_Robustness" / "results" / "phase1_phase2_paired_multiseed_compromise_matched_v1"
ROOT.mkdir(parents=True, exist_ok=True)

BASE_RUNNER = PROJECT / "Ablations" / "Phase2_Hypothesis_Validation" / "runtime" / "run_phase2_hypothesis_ablation.py"
ENGINE_PATH = PROJECT / "Source" / "Project" / "low_canopy_growthloss_ablation_runner.py"
ADAPTER_PATH = PROJECT / "Source" / "Project" / "aoi_masked_phase2_adapter.py"
CONFIG_PATH = PROJECT / "Source" / "Project" / "b4_c15_config.py"

SEEDS = [0, 7, 123, 2024]
SITES_TO_RUN = ["agadir", "ifran", "maamoura"]
RUN_TRAINING = True
FREEZE_REGISTRY = True
AUTHORIZE_ONE_TIME_TEST = True

# Corrective audit: identical composite checkpoint rule on both sides.
PHASE1_CHECKPOINT_NAME = "best_compromise.ckpt"
PHASE2_CHECKPOINT_NAME = "best_compromise.ckpt"
EXPECTED_TEST_N = {"ifran": 5076, "maamoura": 1799, "agadir": 6125}
FINAL_SPEC = {
    "ifran": {"D": 5.0, "K": 2, "lambda_temp": 0.0, "huber_delta": 3.0},
    "maamoura": {"D": 2.0, "K": 2, "lambda_temp": 0.0, "huber_delta": 3.0},
    "agadir": {"D": 3.0, "K": 3, "lambda_temp": 0.10, "huber_delta": 3.0},
}

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

def support_fingerprint(ids):
    payload = "\n".join(sorted(map(str, ids))).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

base = load_module("paired_ms_base", BASE_RUNNER)
cfgmod = load_module("paired_ms_config", CONFIG_PATH)
print("Requested paired runs:", len(SEEDS) * len(SITES_TO_RUN))
print("Seeds:", SEEDS)
print("Training enabled:", RUN_TRAINING, "TEST authorised:", AUTHORIZE_ONE_TIME_TEST)


In [ ]:
def phase1_run_dir(forest, seed):
    cfg = cfgmod.SITES[forest]
    run_name = f"{cfg.run_name}_MULTISEED_SEED{seed}"
    return cfg.runs_root / cfg.catalog_root.name / run_name

parents = []
for forest in SITES_TO_RUN:
    for seed in SEEDS:
        run_dir = phase1_run_dir(forest, seed)
        checkpoint = run_dir / "checkpoints" / PHASE1_CHECKPOINT_NAME
        parents.append({
            "forest": forest,
            "seed": seed,
            "phase1_run_dir": str(run_dir),
            "phase1_checkpoint": str(checkpoint),
            "phase1_checkpoint_exists": checkpoint.is_file(),
            "phase1_checkpoint_sha256": sha256(checkpoint) if checkpoint.is_file() else None,
        })

parents = pd.DataFrame(parents)
display(parents)
if not parents["phase1_checkpoint_exists"].all():
    missing = parents.loc[~parents["phase1_checkpoint_exists"],
                          ["forest", "seed", "phase1_checkpoint"]]
    raise FileNotFoundError("Missing Phase 1 best_compromise parents:\n" + missing.to_string(index=False))
if parents["phase1_checkpoint_sha256"].nunique() != len(parents):
    raise RuntimeError("Expected 12 distinct Phase 1 checkpoint hashes.")
parents.to_csv(ROOT / "00_phase1_parent_registry.csv", index=False)


In [ ]:
def prepare(forest, seed, parent_path, parent_sha):
    engine = load_module(f"paired_engine_{forest}_{seed}", ENGINE_PATH)
    adapter = load_module(f"paired_adapter_{forest}_{seed}", ADAPTER_PATH)
    cfg = engine.FORESTS[forest]
    cfg["parent"] = Path(parent_path)
    cfg["parent_sha"] = parent_sha
    cfg["ablation_family"] = "Phase1_Phase2_Paired_MultiSeed_CompromiseMatched_v1"

    spec = FINAL_SPEC[forest]
    candidate = f"PAIRED_COMPROMISE_MATCHED_SEED{seed}"
    candidate_spec = {
        "drop_m": spec["D"],
        "K": spec["K"],
        "lambda_growth": spec["lambda_temp"],
        "huber_delta": spec["huber_delta"],
    }
    if forest == "ifran":
        engine.IFRAN_CANDIDATES = {candidate: candidate_spec}
    else:
        engine.LOW_CANOPY_CANDIDATES = {candidate: candidate_spec}

    def roots(_forest):
        return ROOT / "runs" / forest / "training", ROOT / "runs" / forest / "reports"

    engine.roots = roots
    modules = adapter.prepare_aoi_masked_modules(engine)
    return engine, modules, cfg, candidate, candidate_spec

plan_rows = []
for row in parents.itertuples(index=False):
    engine, _, _, candidate, spec = prepare(
        row.forest, int(row.seed), row.phase1_checkpoint, row.phase1_checkpoint_sha256
    )
    run_dir = engine.run_dir(row.forest, candidate)
    plan_rows.append({
        "forest": row.forest,
        "seed": int(row.seed),
        "candidate": candidate,
        "phase1_checkpoint": row.phase1_checkpoint,
        "phase1_checkpoint_sha256": row.phase1_checkpoint_sha256,
        "phase2_run_dir": str(run_dir),
        **spec,
    })

plan = pd.DataFrame(plan_rows)
plan.to_csv(ROOT / "01_prespecified_paired_run_plan.csv", index=False)
display(plan)


In [ ]:
def train_one(row):
    forest, seed = row.forest, int(row.seed)
    engine, modules, cfg, candidate, spec = prepare(
        forest, seed, row.phase1_checkpoint, row.phase1_checkpoint_sha256
    )
    run_dir = Path(row.phase2_run_dir)
    shots, records = base.harmonize(engine, modules, forest)
    train_ds = modules["B4SequenceCropDataset"](
        records["train"], shots, crop_size=96, samples_per_epoch=264,
        drop_channels=(), seed=seed, center_on_gedi=True,
        balanced_height_anchors=True, height_bins=cfg["height_bins"],
    )
    val_ds = modules["B4SequenceCropDataset"](
        records["val"], shots, crop_size=96,
        samples_per_epoch=max(132, 4 * len(records["val"])),
        drop_channels=(), seed=seed + 10000, center_on_gedi=True,
        balanced_height_anchors=False, height_bins=cfg["height_bins"],
    )
    model = engine.fresh_model(cfg, modules)
    modules["train"](
        model=model, train_dataset=train_ds, val_dataset=val_ds,
        official_repo=engine.OFFICIAL_REPO, phase1_checkpoint=cfg["parent"],
        run_dir=run_dir, device=engine.DEVICE, seed=seed, batch_size=1,
        max_steps=cfg["max_steps"], val_every_steps=66, patience_evals=20,
        learning_rate=1e-4, weight_decay=5e-3,
        lambda_growth=spec["lambda_growth"], lambda_slope=0, lambda_std=0,
        lambda_bias=0, lambda_anti_zero=0, supervised_loss_name="huber",
        huber_delta=spec["huber_delta"], height_weight_mode="none",
        slope_min=0, slope_max=2, disturbance_indicator=-1,
        disturbance_rule="persistent_running_max",
        persistent_drop_m=spec["drop_m"],
        persistent_required_consecutive_flags=spec["K"],
        full_disturbance_window=True, min_height=cfg["eval_min"],
        max_height=cfg["eval_max"], checkpoint_min_slope=0,
        checkpoint_min_std_ratio=0, checkpoint_max_std_ratio=2,
        checkpoint_max_abs_bias=5, warmup_cycles=3, plateau_patience=8,
        plateau_factor=0.5, lr_min=1e-6, grad_clip=1,
        allow_resume=True, reuse_completed=True,
    )
    del model, train_ds, val_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if RUN_TRAINING:
    for row in plan.itertuples(index=False):
        print("TRAIN", row.forest, row.seed, flush=True)
        train_one(row)
else:
    print("Training disabled. Set RUN_TRAINING=True only for the controlled GPU run.")


In [ ]:
registry_rows = []
for row in plan.itertuples(index=False):
    checkpoint = Path(row.phase2_run_dir) / "checkpoints" / PHASE2_CHECKPOINT_NAME
    rec = dict(row._asdict())
    rec.update({
        "phase2_checkpoint": str(checkpoint),
        "complete": checkpoint.is_file(),
        "selection_split": "VAL",
        "test_used_for_selection": False,
    })
    if checkpoint.is_file():
        state = torch.load(checkpoint, map_location="cpu", weights_only=False)
        metrics = state.get("metrics", {})
        rec.update({
            "phase2_checkpoint_sha256": sha256(checkpoint),
            "val_n": metrics.get("n"),
            "val_mae": metrics.get("mae"),
            "val_rmse": metrics.get("rmse"),
            "val_r2": metrics.get("r2"),
            "val_bias": metrics.get("bias"),
            "val_slope": metrics.get("slope"),
            "val_std_ratio": metrics.get("std_ratio"),
        })
    registry_rows.append(rec)

registry = pd.DataFrame(registry_rows)
all_complete = bool(registry["complete"].all())
display(registry)
if FREEZE_REGISTRY:
    registry.to_csv(ROOT / "02_frozen_paired_checkpoint_registry.csv", index=False)
    protocol = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "seeds": SEEDS,
        "forests": SITES_TO_RUN,
        "all_complete": all_complete,
        "phase1_checkpoint_rule": PHASE1_CHECKPOINT_NAME,
        "phase2_checkpoint_rule": PHASE2_CHECKPOINT_NAME + " selected on VAL only",
        "test_opened": False,
    }
    (ROOT / "03_freeze_protocol.json").write_text(json.dumps(protocol, indent=2), encoding="utf-8")
print("ALL PAIRED PHASE 2 CHECKPOINTS FROZEN:", all_complete)
if AUTHORIZE_ONE_TIME_TEST and not all_complete:
    raise RuntimeError("TEST blocked: every prespecified paired checkpoint must exist first.")


In [ ]:
def regression_metrics(y, prediction):
    y = np.asarray(y, dtype=float)
    prediction = np.asarray(prediction, dtype=float)
    valid = np.isfinite(y) & np.isfinite(prediction)
    y, prediction = y[valid], prediction[valid]
    error = prediction - y
    slope = np.polyfit(y, prediction, 1)[0] if len(y) > 1 else np.nan
    ss_tot = np.sum((y - y.mean()) ** 2)
    return {
        "n": len(y),
        "mae": np.mean(np.abs(error)),
        "rmse": np.sqrt(np.mean(error ** 2)),
        "bias": np.mean(error),
        "r2": 1 - np.sum(error ** 2) / ss_tot if ss_tot > 0 else np.nan,
        "slope": slope,
        "std_ratio": prediction.std() / y.std() if y.std() > 0 else np.nan,
    }

def phase2_test(row):
    forest, seed = row.forest, int(row.seed)
    engine, modules, cfg, candidate, _ = prepare(
        forest, seed, row.phase1_checkpoint, row.phase1_checkpoint_sha256
    )
    checkpoint = Path(row.phase2_checkpoint)
    state = torch.load(checkpoint, map_location="cpu", weights_only=False)
    _, shots, records = engine.build_data(forest, modules, include_test=True)
    model = engine.fresh_model(cfg, modules)
    model.prediction_head.load_state_dict(state["prediction_head"], strict=True)
    model.eval()
    _, nearest = modules["evaluate_full_patch_temporal_nearest"](
        model=model, records=records["test"], shots=shots, device=engine.DEVICE,
        split="test", drop_channels=(), min_height=cfg["eval_min"],
        max_height=cfg["eval_max"], progress_every=1,
    )
    nearest["aux_shot_uid"] = nearest["aux_shot_uid"].astype(str)
    if len(nearest) != EXPECTED_TEST_N[forest] or not nearest["aux_shot_uid"].is_unique:
        raise RuntimeError(f"Phase 2 support mismatch for {forest}/seed {seed}")
    return nearest

paired_rows = []
if AUTHORIZE_ONE_TIME_TEST:
    for row in registry.itertuples(index=False):
        paired_predictions = phase2_test(row)
        required = {"aux_shot_uid", "rh95", "pred_off_reference", "pred_on_growthloss"}
        missing = required.difference(paired_predictions.columns)
        if missing:
            raise RuntimeError(f"Missing paired prediction columns: {sorted(missing)}")
        p1_ids = set(paired_predictions["aux_shot_uid"])

        # Both predictions come from one evaluator call on exactly the same rows:
        # pred_off_reference = frozen Phase 1 best_compromise parent;
        # pred_on_growthloss = its trained Phase 2 residual refinement.
        p1m = regression_metrics(
            paired_predictions["rh95"], paired_predictions["pred_off_reference"]
        )
        p2m = regression_metrics(
            paired_predictions["rh95"], paired_predictions["pred_on_growthloss"]
        )
        rec = {
            "forest": row.forest,
            "seed": int(row.seed),
            "support_n": len(p1_ids),
            "support_sha256": support_fingerprint(p1_ids),
            "phase1_checkpoint_sha256": row.phase1_checkpoint_sha256,
            "phase2_checkpoint_sha256": row.phase2_checkpoint_sha256,
        }
        for metric in ("mae", "rmse", "bias", "r2", "slope", "std_ratio"):
            rec[f"phase1_{metric}"] = p1m[metric]
            rec[f"phase2_{metric}"] = p2m[metric]
            rec[f"delta_{metric}"] = p2m[metric] - p1m[metric]
        paired_rows.append(rec)

    paired = pd.DataFrame(paired_rows)
    paired.to_csv(ROOT / "04_paired_test_metrics_by_seed.csv", index=False)
    protocol_path = ROOT / "03_freeze_protocol.json"
    protocol = json.loads(protocol_path.read_text(encoding="utf-8"))
    protocol.update({"test_opened": True, "test_opened_utc": datetime.now(timezone.utc).isoformat()})
    protocol_path.write_text(json.dumps(protocol, indent=2), encoding="utf-8")
else:
    output = ROOT / "04_paired_test_metrics_by_seed.csv"
    paired = pd.read_csv(output) if output.exists() else pd.DataFrame()
    print("TEST remains closed. Set AUTHORIZE_ONE_TIME_TEST=True only after registry freeze.")

display(paired)


In [ ]:
if paired.empty:
    print("No paired TEST table yet. Complete and freeze all runs before opening TEST.")
else:
    summary_rows = []
    for forest, group in paired.groupby("forest"):
        rec = {"forest": forest, "n_seeds": group["seed"].nunique(), "support_n": group["support_n"].iloc[0]}
        for metric in ("mae", "rmse", "bias", "r2", "slope", "std_ratio"):
            delta = group[f"delta_{metric}"]
            rec[f"delta_{metric}_mean"] = delta.mean()
            rec[f"delta_{metric}_sd"] = delta.std(ddof=1)
            rec[f"delta_{metric}_min"] = delta.min()
            rec[f"delta_{metric}_max"] = delta.max()
            rec[f"delta_{metric}_negative_seeds"] = int((delta < 0).sum())
            rec[f"delta_{metric}_positive_seeds"] = int((delta > 0).sum())
        summary_rows.append(rec)
    paired_summary = pd.DataFrame(summary_rows)
    paired_summary.to_csv(ROOT / "05_paired_delta_summary_mean_sd.csv", index=False)
    display(paired_summary)
    print("Interpret mean +/- SD and sign consistency descriptively; n=4 is too small for strong inferential claims.")
